In [ ]:

import pandas as pd

proteomics = pd.read_csv(
    "ProCan-DepMapSanger_protein_matrix_8498_averaged.txt",
    sep="\t"
)

mapping = pd.read_csv(
    "ProCan-DepMapSanger_mapping_file_averaged.txt",
    sep="\t"
)

print("Proteomics:", proteomics.shape)
print("Mapping:", mapping.shape)


Proteomics: (949, 8499)
Mapping: (949, 6)


In [ ]:

print("Proteomics cell lines:",
      proteomics["Project_Identifier"].nunique())

print("Mapping cell lines:",
      mapping["Project_Identifier"].nunique())

print("Different tissue types:",
      mapping["Tissue_type"].nunique())


Proteomics cell lines: 949
Mapping cell lines: 949
Different tissue types: 28


In [ ]:

protein_ids = set(proteomics["Project_Identifier"])
mapping_ids = set(mapping["Project_Identifier"])

print("IDs in proteomics but not mapping:",
      len(protein_ids - mapping_ids))

print("IDs in mapping but not proteomics:",
      len(mapping_ids - protein_ids))

print("Matching IDs:",
      len(protein_ids & mapping_ids))


IDs in proteomics but not mapping: 0
IDs in mapping but not proteomics: 0
Matching IDs: 949


In [ ]:

# Select only the protein measurement columns
protein_values = proteomics.drop(
    columns=["Project_Identifier"]
)

total = protein_values.size
missing = protein_values.isna().sum().sum()
present = total - missing

print("Total measurements:", total)
print("Present measurements:", present)
print("Missing measurements:", missing)
print("Percent missing:", round(missing / total * 100, 2))

print("Proteins entirely missing:",
      (protein_values.notna().sum() == 0).sum())


Total measurements: 8064602
Present measurements: 4942859
Missing measurements: 3121743
Percent missing: 38.71
Proteins entirely missing: 1


In [ ]:

gdsc1 = pd.read_csv("GDSC1 - cell lines.csv")

print("Dataset shape:", gdsc1.shape)
print("Unique cell-line names:", gdsc1["cellLine"].nunique())
print("Unique database IDs:", gdsc1["id"].nunique())
print("Missing cell-line names:", gdsc1["cellLine"].isna().sum())

gdsc1.head()


Dataset shape: (1104, 4)
Unique cell-line names: 1104
Unique database IDs: 1104
Missing cell-line names: 0


,dataset,id,cell_uid,cellLine
0,GDSC1,2,NCI-H29513862021_,NCI-H295
1,GDSC1,3,SW13_1529_2019,SW13
2,GDSC1,6,EGI1_327_2019,EGI-1
3,GDSC1,7,ETK1_348_2019,ETK-1
4,GDSC1,8,HuCCT1_635_2019,HuCC-T1


In [ ]:

# Convert cell-line names to a common format
def normalize_name(name):
    return str(name).strip().upper().replace("-", "").replace(" ", "")

proteomics_names = set(
    mapping["Cell_line"].map(normalize_name)
)

gdsc1_names = set(
    gdsc1["cellLine"].dropna().map(normalize_name)
)

overlap = proteomics_names & gdsc1_names

print("Proteomics cell lines:", len(proteomics_names))
print("GDSC1 cell lines:", len(gdsc1_names))
print("Initial name-based overlap:", len(overlap))


Proteomics cell lines: 947
GDSC1 cell lines: 1103
Initial name-based overlap: 864


In [ ]:

mapping["normalized_name"] = mapping["Cell_line"].map(normalize_name)

duplicates = mapping[
    mapping.duplicated("normalized_name", keep=False)
]

print(duplicates[
    ["Cell_line", "SIDM", "Project_Identifier", "normalized_name"]
].to_string(index=False))


Cell_line      SIDM Project_Identifier normalized_name
       TT SIDM00484       SIDM00484;TT              TT
    KM-H2 SIDM01018    SIDM01018;KM-H2            KMH2
    KMH-2 SIDM00619    SIDM00619;KMH-2            KMH2
      T-T SIDM00322      SIDM00322;T-T              TT


In [ ]:

def normalize_name_safe(name):
    return str(name).strip().upper()

proteomics_names = set(
    mapping["Cell_line"].dropna().map(normalize_name_safe)
)

gdsc1_names = set(
    gdsc1["cellLine"].dropna().map(normalize_name_safe)
)

overlap_safe = proteomics_names & gdsc1_names

print("Proteomics unique names:", len(proteomics_names))
print("GDSC1 unique names:", len(gdsc1_names))
print("Conservative overlap:", len(overlap_safe))


Proteomics unique names: 949
GDSC1 unique names: 1104
Conservative overlap: 764


In [ ]:

unmatched = sorted(proteomics_names - gdsc1_names)

print("Unmatched proteomics names:", len(unmatched))
print("\nFirst 30 unmatched names:")

for name in unmatched[:30]:
    print(name)


Unmatched proteomics names: 185

First 30 unmatched names:
23132-87
639-V
647-V
786-0
A172
A204
A253
A3-KAW
A375
A388
A4-FUK
A427
A431
A498
A549
A673
A704
AMO-1
BE2-M17
BICR22
BICR31
BICR78
C-33-A
C32
C3A
CA-SKI
COLO-205
COLO-320-HSR
COLO-668
COLO-678


In [ ]:

# Create a simplified name for finding potential matches
def simplify_name(name):
    return str(name).upper().replace("-", "").replace(" ", "")

# Create copies without changing the original data
protein_check = mapping[["Cell_line", "SIDM"]].copy()
gdsc_check = gdsc1[["cellLine", "cell_uid"]].copy()

protein_check["simple"] = protein_check["Cell_line"].map(simplify_name)
gdsc_check["simple"] = gdsc_check["cellLine"].map(simplify_name)

# Only examine previously unmatched proteomics names
unmatched_df = protein_check[
    protein_check["Cell_line"].map(normalize_name_safe).isin(unmatched)
]

# Find potential matches
possible_matches = unmatched_df.merge(
    gdsc_check,
    on="simple",
    how="inner"
)

print("Possible matches:", len(possible_matches))

display(
    possible_matches[
        ["Cell_line", "SIDM", "cellLine", "cell_uid"]
    ].head(30)
)


Possible matches: 102


,Cell_line,SIDM,cellLine,cell_uid
0,MV-4-11,SIDM00657,MV4-11,MV411_972_2019
1,COLO-684,SIDM00142,COLO 684,COLO684_229_2019
2,Hs-766T,SIDM00664,Hs 766T,Hs766T_596_2019
3,PA-TU-8902,SIDM00455,PaTu 8902,PATU8902_1236_2019
4,OCI-AML2,SIDM00446,OCI-AML-2,OCIAML2_1183_2019
5,WM-115,SIDM01000,WM115,WM115_1662_2019
6,LS-513,SIDM00677,LS513,LS513_864_2019
7,Ca-Ski,SIDM00923,Ca Ski,CaSki_158_2019
8,WIL2-NS,SIDM01102,WIL2 NS,WIL2NS_1661_2019
9,HuCCT1,SIDM00587,HuCC-T1,HuCCT1_635_2019


In [ ]:

# Count distinct cell-line IDs for each simplified name
protein_counts = protein_check.groupby("simple")["SIDM"].nunique()
gdsc_counts = gdsc_check.groupby("simple")["cell_uid"].nunique()

possible_matches["protein_candidates"] = (
    possible_matches["simple"].map(protein_counts)
)

possible_matches["gdsc_candidates"] = (
    possible_matches["simple"].map(gdsc_counts)
)

# Identify potential ambiguities
ambiguous = possible_matches[
    (possible_matches["protein_candidates"] > 1) |
    (possible_matches["gdsc_candidates"] > 1)
]

print("Total candidate pairs:", len(possible_matches))
print("Ambiguous candidate pairs:", len(ambiguous))

display(
    ambiguous[
        ["Cell_line", "SIDM", "cellLine",
         "cell_uid", "protein_candidates", "gdsc_candidates"]
    ]
)


Total candidate pairs: 102
Ambiguous candidate pairs: 1


,Cell_line,SIDM,cellLine,cell_uid,protein_candidates,gdsc_candidates
70,T-T,SIDM00322,TT,TT_1606_2019,2,1


In [ ]:

# Keep only one-to-one formatting-based candidates
safe_candidates = possible_matches[
    (possible_matches["protein_candidates"] == 1) &
    (possible_matches["gdsc_candidates"] == 1)
].copy()

print("Initial direct matches:", len(overlap_safe))
print("Unambiguous additional candidates:", len(safe_candidates))

print("Provisional combined overlap:",
      len(overlap_safe) + safe_candidates["SIDM"].nunique())


Initial direct matches: 764
Unambiguous additional candidates: 101
Provisional combined overlap: 865


In [ ]:

# Direct matches
direct_matches = mapping[
    mapping["Cell_line"].map(normalize_name_safe).isin(overlap_safe)
][["Cell_line", "SIDM"]].copy()

direct_matches["match_type"] = "Direct name match"

# Additional formatting candidates
additional_matches = safe_candidates[
    ["Cell_line", "SIDM"]
].copy()

additional_matches["match_type"] = "Formatting candidate"

# Combine
combined = pd.concat(
    [direct_matches, additional_matches],
    ignore_index=True
)

combined = combined.drop_duplicates(subset=["SIDM"])

print("Unique matched proteomics IDs:", combined["SIDM"].nunique())

combined.to_csv(
    "Proteomics_GDSC1_provisional_overlap.csv",
    index=False
)


Unique matched proteomics IDs: 865


In [ ]:

gdsc2 = pd.read_csv("GDSC2 - cell lines.csv")

print("GDSC2 shape:", gdsc2.shape)
print("Unique cell lines:", gdsc2["cellLine"].nunique())

# Compare GDSC1 and GDSC2, excluding dataset labels
columns = ["id", "cell_uid", "cellLine"]

gdsc1_sorted = gdsc1[columns].sort_values(columns).reset_index(drop=True)
gdsc2_sorted = gdsc2[columns].sort_values(columns).reset_index(drop=True)

print("Identical cell-line records:",
      gdsc1_sorted.equals(gdsc2_sorted))


GDSC2 shape: (1104, 4)
Unique cell lines: 1104
Identical cell-line records: True


In [ ]:

import pandas as pd
import gzip
import csv
from collections import Counter

filename = "gene_attribute_matrix (1).txt.gz"

with gzip.open(filename, "rt") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)
    tissue_row = next(reader)
    gene_header = next(reader)

cell_names = header[3:]

print("Cell-line columns:", len(cell_names))
print("Unique cell-line names:", len(set(cell_names)))
print("Duplicate names:",
      [name for name, count in Counter(cell_names).items()
       if count > 1])


Cell-line columns: 1037
Unique cell-line names: 1035
Duplicate names: ['NCIH292', 'TT']


In [ ]:

genes = []
value_counts = Counter()

with gzip.open(filename, "rt") as f:
    reader = csv.reader(f, delimiter="\t")

    for _ in range(3):
        next(reader)

    for row in reader:
        genes.append(row[0])
        value_counts.update(row[3:])

print("Gene rows:", len(genes))
print("Unique gene names:", len(set(genes)))
print("Value frequencies:", value_counts)


Gene rows: 18025
Unique gene names: 18025
Value frequencies: Counter({'-0.000000': 9066276, '0.000000': 8874426, '1.000000': 420474, '-1.000000': 330749})


In [ ]:

expression_names = {
    normalize_name_safe(name)
    for name in cell_names
}

protein_names = {
    normalize_name_safe(name)
    for name in mapping["Cell_line"].dropna()
}

expression_overlap = expression_names & protein_names

print("Expression cell lines:", len(expression_names))
print("Proteomics cell lines:", len(protein_names))
print("Direct name overlap:", len(expression_overlap))


Expression cell lines: 1035
Proteomics cell lines: 949
Direct name overlap: 132


In [ ]:

print("First 20 expression cell-line names:")
print(cell_names[:20])

print("\nFirst 20 proteomics cell-line names:")
print(mapping["Cell_line"].head(20).tolist())

print("\nExamples of matched names:")
print(sorted(expression_overlap)[:20])


First 20 expression cell-line names:
['CHL1', 'HMCB', 'HS852T', 'HS695T', 'A101D', 'HS294T', 'SNU466', 'SKMEL31', 'SKMEL24', 'WM115', 'COLO818', 'COLO800', 'HT144', 'HS939T', 'RVH421', 'C32', 'UACC62', 'COLO679', 'COLO829', 'COLO849']

First 20 proteomics cell-line names:
['BC-1', 'L-363', 'EoL-1-cell', 'NCI-H727', 'MV-4-11', 'HSC-39', 'RPMI-8402', 'MDA-MB-453', 'ES8', 'NKM-1', 'KURAMOCHI', 'UACC-62', 'Ca9-22', 'COLO-684', 'SW13', 'SF539', 'MDA-MB-175-VII', 'NCI-H524', 'BxPC-3', 'NCI-H847']

Examples of matched names:
['22RV1', '5637', '697', '8305C', '8505C', 'A172', 'A204', 'A2058', 'A253', 'A2780', 'A375', 'A498', 'A549', 'A673', 'A704', 'ACHN', 'AGS', 'AU565', 'BECKER', 'BEN']


In [ ]:

import re
from collections import Counter

# Standardize punctuation for candidate matching
def simplify_name(name):
    return re.sub(r"[\s\-_]", "", str(name).upper())

expression_simple = [simplify_name(n) for n in cell_names]
protein_simple = mapping["Cell_line"].map(simplify_name)

# Count possible naming collisions
expr_counts = Counter(expression_simple)
protein_counts = Counter(protein_simple)

# Only consider names with one unique proteomics ID
# and one unique expression name
expr_unique = set(expression_simple)
protein_unique = set(protein_simple)

candidate_overlap = expr_unique & protein_unique

safe_overlap = {
    name for name in candidate_overlap
    if protein_counts[name] == 1
    and expr_counts[name] == 1
}

ambiguous_overlap = candidate_overlap - safe_overlap

print("Direct name overlap:", len(expression_overlap))
print("Formatting-compatible, unique matches:", len(safe_overlap))
print("Ambiguous normalized matches:", len(ambiguous_overlap))
print("Names unmatched after normalization:",
      len(protein_unique - expr_unique))


Direct name overlap: 132
Formatting-compatible, unique matches: 605
Ambiguous normalized matches: 3
Names unmatched after normalization: 339


In [ ]:

print("Ambiguous normalized names:")
for name in sorted(ambiguous_overlap):
    print(name)

print("\nFirst 30 unmatched proteomics names:")
for name in sorted(protein_unique - expr_unique)[:30]:
    print(name)


Ambiguous normalized names:
KMH2
NCIH292
TT

First 30 unmatched proteomics names:
201T
451LU
7860
A388
A427
A431
ALLPO
ASH3
ATN1
BALL1
BB30HNC
BB65RCC
BC1
BC3
BE13
BE2M17
BICR78
BONNA12
BPH1
C33A
C4I
CA922
CAL39
CAL72
CAR1
CASKI
CCRFCEM
CHSA0011
CHSA0108
CHSA8926


In [ ]:

# Save formatting-compatible matches
expression_matches = mapping[
    mapping["Cell_line"].map(simplify_name).isin(safe_overlap)
][["Cell_line", "SIDM", "Project_Identifier"]].copy()

expression_matches["expression_name"] = (
    expression_matches["Cell_line"]
    .map(lambda x: next(
        (n for n in cell_names
         if simplify_name(n) == simplify_name(x)),
        None
    ))
)

expression_matches.to_csv(
    "Proteomics_CCLE_expression_provisional_overlap.csv",
    index=False
)

print("Matched cell lines:", len(expression_matches))
print("Unique SIDM IDs:", expression_matches["SIDM"].nunique())

display(expression_matches.head())


Matched cell lines: 605
Unique SIDM IDs: 605


,Cell_line,SIDM,Project_Identifier,expression_name
1,L-363,SIDM00312,SIDM00312;L-363,L363
3,NCI-H727,SIDM01119,SIDM01119;NCI-H727,NCIH727
4,MV-4-11,SIDM00657,SIDM00657;MV-4-11,MV411
6,RPMI-8402,SIDM00403,SIDM00403;RPMI-8402,RPMI8402
7,MDA-MB-453,SIDM00272,SIDM00272;MDA-MB-453,MDAMB453


In [ ]:

import gzip
import csv
from collections import Counter

filename = "gene_attribute_matrix (2).txt.gz"

with gzip.open(filename, "rt") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)
    tissue_row = next(reader)
    gene_header = next(reader)

mutation_cell_names = header[3:]

print("Cell-line columns:", len(mutation_cell_names))
print("Unique cell lines:", len(set(mutation_cell_names)))
print("Duplicate names:", [
    name for name, count in Counter(mutation_cell_names).items()
    if count > 1
])


Cell-line columns: 904
Unique cell lines: 904
Duplicate names: []


In [ ]:

mutation_genes = []
mutation_values = Counter()

with gzip.open(filename, "rt") as f:
    reader = csv.reader(f, delimiter="\t")

    for _ in range(3):
        next(reader)

    for row in reader:
        mutation_genes.append(row[0])
        mutation_values.update(row[3:])

print("Gene rows:", len(mutation_genes))
print("Unique genes:", len(set(mutation_genes)))
print("Value frequencies:", mutation_values)


Gene rows: 1667
Unique genes: 1667
Value frequencies: Counter({'0.000000': 1401047, '1.000000': 105921})


In [ ]:

# Direct matching
mutation_names = {
    normalize_name_safe(name)
    for name in mutation_cell_names
}

protein_names = {
    normalize_name_safe(name)
    for name in mapping["Cell_line"].dropna()
}

mutation_direct_overlap = mutation_names & protein_names

print("Mutation cell lines:", len(mutation_names))
print("Proteomics cell lines:", len(protein_names))
print("Direct name overlap:", len(mutation_direct_overlap))


Mutation cell lines: 904
Proteomics cell lines: 949
Direct name overlap: 125


In [ ]:

# Identify formatting-compatible matches
mutation_simple = [simplify_name(n) for n in mutation_cell_names]
protein_simple = mapping["Cell_line"].map(simplify_name)

mutation_counts = Counter(mutation_simple)
protein_counts = Counter(protein_simple)

candidate_overlap = set(mutation_simple) & set(protein_simple)

safe_mutation_overlap = {
    name for name in candidate_overlap
    if mutation_counts[name] == 1
    and protein_counts[name] == 1
}

ambiguous_mutation = candidate_overlap - safe_mutation_overlap

print("Formatting-compatible unique matches:",
      len(safe_mutation_overlap))

print("Ambiguous normalized matches:",
      len(ambiguous_mutation))

print("Ambiguous names:", sorted(ambiguous_mutation))


Formatting-compatible unique matches: 555
Ambiguous normalized matches: 2
Ambiguous names: ['KMH2', 'TT']


In [ ]:

mutation_matches = mapping[
    mapping["Cell_line"].map(simplify_name).isin(safe_mutation_overlap)
][["Cell_line", "SIDM", "Project_Identifier"]].copy()

# Link each proteomics line to its mutation dataset name
mutation_lookup = {
    simplify_name(name): name
    for name in mutation_cell_names
    if simplify_name(name) in safe_mutation_overlap
}

mutation_matches["mutation_name"] = (
    mutation_matches["Cell_line"]
    .map(lambda name: mutation_lookup.get(simplify_name(name)))
)

mutation_matches.to_csv(
    "Proteomics_CCLE_mutation_provisional_overlap.csv",
    index=False
)

print("Matched cell lines:", len(mutation_matches))
print("Unique SIDM IDs:", mutation_matches["SIDM"].nunique())

display(mutation_matches.head())


Matched cell lines: 555
Unique SIDM IDs: 555


,Cell_line,SIDM,Project_Identifier,mutation_name
1,L-363,SIDM00312,SIDM00312;L-363,L363
3,NCI-H727,SIDM01119,SIDM01119;NCI-H727,NCIH727
4,MV-4-11,SIDM00657,SIDM00657;MV-4-11,MV411
6,RPMI-8402,SIDM00403,SIDM00403;RPMI-8402,RPMI8402
7,MDA-MB-453,SIDM00272,SIDM00272;MDA-MB-453,MDAMB453


In [ ]:

import gzip
import csv
from collections import Counter

filename = "gene_attribute_matrix (3).txt.gz"

with gzip.open(filename, "rt") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)
    tissue_row = next(reader)
    gene_header = next(reader)

cnv_cell_names = header[3:]

print("Cell-line columns:", len(cnv_cell_names))
print("Unique cell-line names:", len(set(cnv_cell_names)))

print("Duplicate names:", {
    name: count
    for name, count in Counter(cnv_cell_names).items()
    if count > 1
})


Cell-line columns: 1043
Unique cell-line names: 1040
Duplicate names: {'TT': 2, 'KELLY': 3}


In [ ]:

cnv_genes = []
cnv_values = Counter()

with gzip.open(filename, "rt") as f:
    reader = csv.reader(f, delimiter="\t")

    for _ in range(3):
        next(reader)

    for row in reader:
        cnv_genes.append(row[0])
        cnv_values.update(row[3:])

print("Gene rows:", len(cnv_genes))
print("Unique genes:", len(set(cnv_genes)))
print("Value frequencies:", cnv_values)

print("Blank entries:", cnv_values.get("", 0))


Gene rows: 23264
Unique genes: 23264
Value frequencies: Counter({'-0.000000': 11583646, '0.000000': 11529572, '-1.000000': 580657, '1.000000': 570477})
Blank entries: 0


In [ ]:

cnv_names = {
    normalize_name_safe(name)
    for name in cnv_cell_names
}

protein_names = {
    normalize_name_safe(name)
    for name in mapping["Cell_line"].dropna()
}

cnv_direct_overlap = cnv_names & protein_names

print("CNV unique cell lines:", len(cnv_names))
print("Proteomics cell lines:", len(protein_names))
print("Direct name overlap:", len(cnv_direct_overlap))


CNV unique cell lines: 1040
Proteomics cell lines: 949
Direct name overlap: 139


In [ ]:

cnv_simple = [simplify_name(n) for n in cnv_cell_names]
protein_simple = mapping["Cell_line"].map(simplify_name)

cnv_counts = Counter(cnv_simple)
protein_counts = Counter(protein_simple)

candidate_overlap = set(cnv_simple) & set(protein_simple)

safe_cnv_overlap = {
    name for name in candidate_overlap
    if cnv_counts[name] == 1
    and protein_counts[name] == 1
}

ambiguous_cnv = candidate_overlap - safe_cnv_overlap

print("Formatting-compatible unique matches:",
      len(safe_cnv_overlap))

print("Ambiguous normalized matches:",
      len(ambiguous_cnv))

print("Ambiguous names:", sorted(ambiguous_cnv))


Formatting-compatible unique matches: 615
Ambiguous normalized matches: 3
Ambiguous names: ['KELLY', 'KMH2', 'TT']


In [ ]:

# Create a lookup from simplified CNV names to original names
cnv_lookup = {
    simplify_name(name): name
    for name in cnv_cell_names
    if simplify_name(name) in safe_cnv_overlap
}

# Select proteomics cell lines with safe CNV candidates
cnv_matches = mapping[
    mapping["Cell_line"].map(simplify_name).isin(safe_cnv_overlap)
][["Cell_line", "SIDM", "Project_Identifier"]].copy()

cnv_matches["cnv_name"] = (
    cnv_matches["Cell_line"]
    .map(lambda name: cnv_lookup.get(simplify_name(name)))
)

cnv_matches.to_csv(
    "Proteomics_CCLE_CNV_provisional_overlap.csv",
    index=False
)

print("Matched cell lines:", len(cnv_matches))
print("Unique SIDM IDs:", cnv_matches["SIDM"].nunique())

display(cnv_matches.head())


Matched cell lines: 615
Unique SIDM IDs: 615


,Cell_line,SIDM,Project_Identifier,cnv_name
1,L-363,SIDM00312,SIDM00312;L-363,L363
3,NCI-H727,SIDM01119,SIDM01119;NCI-H727,NCIH727
4,MV-4-11,SIDM00657,SIDM00657;MV-4-11,MV411
6,RPMI-8402,SIDM00403,SIDM00403;RPMI-8402,RPMI8402
7,MDA-MB-453,SIDM00272,SIDM00272;MDA-MB-453,MDAMB453


In [ ]:

# Extract SIDM identifiers from each matched dataset
protein_ids = set(mapping["SIDM"])
expression_ids = set(expression_matches["SIDM"])
mutation_ids = set(mutation_matches["SIDM"])
cnv_ids = set(cnv_matches["SIDM"])

# Find cell lines represented in all four omics layers
all_omics_ids = (
    protein_ids
    & expression_ids
    & mutation_ids
    & cnv_ids
)

print("Proteomics:", len(protein_ids))
print("Expression overlap:", len(expression_ids))
print("Mutation overlap:", len(mutation_ids))
print("CNV overlap:", len(cnv_ids))
print("All four omics overlap:", len(all_omics_ids))

# Save combined sample list
all_omics_matches = mapping[
    mapping["SIDM"].isin(all_omics_ids)
].copy()

all_omics_matches.to_csv(
    "All_Four_Omics_Provisional_Overlap.csv",
    index=False
)

display(all_omics_matches.head())


Proteomics: 949
Expression overlap: 605
Mutation overlap: 555
CNV overlap: 615
All four omics overlap: 551


,Cell_line,SIDM,Project_Identifier,Tissue_type,Cancer_type,Cancer_subtype,normalized_name
1,L-363,SIDM00312,SIDM00312;L-363,Haematopoietic and Lymphoid,Plasma Cell Myeloma,NaN,L363
3,NCI-H727,SIDM01119,SIDM01119;NCI-H727,Lung,Carcinoid Tumour,Lung Carcinoid Tumor,NCIH727
4,MV-4-11,SIDM00657,SIDM00657;MV-4-11,Haematopoietic and Lymphoid,Acute Myeloid Leukemia,Acute Monocytic leukemia,MV411
6,RPMI-8402,SIDM00403,SIDM00403;RPMI-8402,Haematopoietic and Lymphoid,T-Lymphoblastic Leukemia,Childhood T acute lymphoblastic leukemia,RPMI8402
7,MDA-MB-453,SIDM00272,SIDM00272;MDA-MB-453,Breast,Breast Carcinoma,Breast adenocarcinoma,MDAMB453


In [ ]:

# Cell lines provisionally matched to all four omics datasets
all_omics_ids = set(all_omics_matches["SIDM"])

# Cell lines provisionally matched to GDSC
gdsc_ids = set(combined["SIDM"])

# Intersection across four omics layers and GDSC
final_candidate_ids = all_omics_ids & gdsc_ids

print("All four omics cell lines:", len(all_omics_ids))
print("GDSC provisional matches:", len(gdsc_ids))
print("Four omics + GDSC catalog:", len(final_candidate_ids))

# Save the overlapping cell-line records
final_candidates = mapping[
    mapping["SIDM"].isin(final_candidate_ids)
].copy()

final_candidates.to_csv(
    "All_Omics_GDSC_Provisional_Overlap.csv",
    index=False
)

display(final_candidates.head())


All four omics cell lines: 551
GDSC provisional matches: 865
Four omics + GDSC catalog: 526


,Cell_line,SIDM,Project_Identifier,Tissue_type,Cancer_type,Cancer_subtype,normalized_name
1,L-363,SIDM00312,SIDM00312;L-363,Haematopoietic and Lymphoid,Plasma Cell Myeloma,NaN,L363
3,NCI-H727,SIDM01119,SIDM01119;NCI-H727,Lung,Carcinoid Tumour,Lung Carcinoid Tumor,NCIH727
4,MV-4-11,SIDM00657,SIDM00657;MV-4-11,Haematopoietic and Lymphoid,Acute Myeloid Leukemia,Acute Monocytic leukemia,MV411
6,RPMI-8402,SIDM00403,SIDM00403;RPMI-8402,Haematopoietic and Lymphoid,T-Lymphoblastic Leukemia,Childhood T acute lymphoblastic leukemia,RPMI8402
7,MDA-MB-453,SIDM00272,SIDM00272;MDA-MB-453,Breast,Breast Carcinoma,Breast adenocarcinoma,MDAMB453


In [ ]:

datasets = {
    "Gene expression": value_counts,
    "Mutation": mutation_values,
    "Copy number": cnv_values
}

for name, counts in datasets.items():
    total = sum(counts.values())
    blank = counts.get("", 0)

    print(name)
    print("Total entries:", total)
    print("Blank entries:", blank)
    print("Blank percentage:",
          round(blank / total * 100, 2), "%")
    print()


Gene expression
Total entries: 18691925
Blank entries: 0
Blank percentage: 0.0 %

Mutation
Total entries: 1506968
Blank entries: 0
Blank percentage: 0.0 %

Copy number
Total entries: 24264352
Blank entries: 0
Blank percentage: 0.0 %

